# Agent Cars (Phase 4)

This notebook publishes moving car telemetry over MQTT using the workshop Copenhagen node network and reroute-aware routes around blocked roads.

In [7]:
# Cell purpose: Load config and initialize MQTT topics for movement, reroutes, and live traffic inputs.
from simulated_city.config import load_config
from simulated_city.topics import (
    cars_reroute_topic,
    cars_telemetry_topic,
    roadwork_events_topic,
    traffic_congestion_topic,
)

config = load_config()
phase1 = config.simulation.car_rerouting_phase1 if config.simulation else None
if phase1 is None:
    raise ValueError("Missing simulation.car_rerouting_phase1 in config.yaml")

base_blocked_segments = set()
roadwork_blocked_segments = sorted(set(phase1.roadwork.blocked_segment_ids))
telemetry_topic = cars_telemetry_topic(config.mqtt.base_topic)
reroute_topic = cars_reroute_topic(config.mqtt.base_topic)
roadwork_topic = roadwork_events_topic(config.mqtt.base_topic)
congestion_topic = traffic_congestion_topic(config.mqtt.base_topic)

roadwork_events_by_tick = {
    1: {"active": False, "blocked_segment_ids": roadwork_blocked_segments.copy()},
    int(phase1.roadwork.start_tick): {"active": True, "blocked_segment_ids": roadwork_blocked_segments.copy()},
    int(phase1.roadwork.end_tick) + 1: {"active": False, "blocked_segment_ids": roadwork_blocked_segments.copy()},
}
live_congestion_state = {"congested_segment_ids": [], "tick": 0}

print(f"Loaded Phase 4 config: seed={phase1.seed}, max_ticks={phase1.max_ticks}, car_count={phase1.car_count}")
print(f"Base blocked segment IDs (always blocked): {sorted(base_blocked_segments)}")
print(
    f"Roadwork active window from config: ticks {phase1.roadwork.start_tick}-{phase1.roadwork.end_tick} "
    f"for segments={roadwork_blocked_segments}"
)
print(f"Roadwork input topic: {roadwork_topic}")
print(f"Congestion input topic: {congestion_topic}")
print(f"Publish topics: telemetry={telemetry_topic}, reroute={reroute_topic}")

Loaded Phase 4 config: seed=7, max_ticks=600, car_count=10
Base blocked segment IDs (always blocked): []
Roadwork active window from config: ticks 180-300 for segments=[44105317, 733901267]
Roadwork input topic: simulated-city/city/roadwork/events
Congestion input topic: simulated-city/city/traffic/congestion
Publish topics: telemetry=simulated-city/city/cars/telemetry, reroute=simulated-city/city/cars/reroute


In [8]:
# Cell purpose: Build deterministic routes and per-car movement state with staggered starts.
import random

from simulated_city.routing import assign_od_pairs, compute_reroute

fixed_pairs = [(pair.origin, pair.destination) for pair in phase1.od_pairs]
assigned_pairs = assign_od_pairs(phase1.car_count, fixed_pairs)
rng = random.Random(phase1.seed)

segment_id_by_nodes = {}
for segment_id, node_pair in phase1.segment_node_pairs.items():
    node_a, node_b = node_pair
    segment_id_int = int(segment_id)
    segment_id_by_nodes[(node_a, node_b)] = segment_id_int
    segment_id_by_nodes[(node_b, node_a)] = segment_id_int

cars = []
for car_index, (origin, destination) in enumerate(assigned_pairs, start=1):
    planned_route = compute_reroute(
        graph_adjacency=phase1.graph_adjacency,
        origin=origin,
        destination=destination,
        blocked_segments=set(),
        segment_node_pairs=phase1.segment_node_pairs,
    )
    active_route = planned_route
    was_rerouted = False

    start_tick = (car_index - 1) % 6
    speed_kmh = 22 + (car_index % 5) * 4
    segment_progress = 0.15 + 0.70 * rng.random()

    if not active_route:
        status = "waiting"
        current_node = origin
    elif len(active_route) == 1:
        status = "arrived"
        current_node = active_route[0]
        segment_progress = 0.0
    else:
        status = "moving" if start_tick == 0 else "waiting"
        current_node = active_route[0]

    cars.append({
        "car_id": f"car-{car_index:02d}",
        "origin": origin,
        "destination": destination,
        "planned_route": planned_route,
        "route": active_route,
        "route_step_index": 0,
        "segment_progress": segment_progress,
        "start_tick": start_tick,
        "speed_kmh": speed_kmh,
        "current_node": current_node,
        "last_segment_id": None,
        "last_reroute_tick": -999,
        "trip_index": car_index - 1,
        "was_rerouted": was_rerouted,
        "status": status,
    })

print(f"Built {len(cars)} cars with deterministic routes and continuous trip mode.")
print(f"Sample entries: {cars[:3]}")

Built 10 cars with deterministic routes and continuous trip mode.
Sample entries: [{'car_id': 'car-01', 'origin': 'N1', 'destination': 'N6', 'planned_route': ['N1', 'N2', 'N3', 'N6'], 'route': ['N1', 'N2', 'N3', 'N6'], 'route_step_index': 0, 'segment_progress': 0.3766829353832136, 'start_tick': 0, 'speed_kmh': 26, 'current_node': 'N1', 'last_segment_id': None, 'last_reroute_tick': -999, 'trip_index': 0, 'was_rerouted': False, 'status': 'moving'}, {'car_id': 'car-02', 'origin': 'N2', 'destination': 'N6', 'planned_route': ['N2', 'N3', 'N6'], 'route': ['N2', 'N3', 'N6'], 'route_step_index': 0, 'segment_progress': 0.2555944217471513, 'start_tick': 1, 'speed_kmh': 30, 'current_node': 'N2', 'last_segment_id': None, 'last_reroute_tick': -999, 'trip_index': 1, 'was_rerouted': False, 'status': 'waiting'}, {'car_id': 'car-03', 'origin': 'N4', 'destination': 'N3', 'planned_route': ['N4', 'N1', 'N2', 'N3'], 'route': ['N4', 'N1', 'N2', 'N3'], 'route_step_index': 0, 'segment_progress': 0.60565413112

In [ ]:
# Cell purpose: Move cars through main routes and reroute only when active roadwork/congestion blocks them.
import json
import time
from datetime import datetime, timezone

from simulated_city import mqtt
from simulated_city.maplibre_live import resolve_node_lnglat
from simulated_city.mqtt import publish_json_checked
from simulated_city.routing import compute_reroute, route_intersects_blocked_segments

ticks_to_run = min(220, int(phase1.max_ticks))
reroute_cooldown_ticks = int(phase1.routing.reroute_cooldown_ticks)
tick_sleep_s = 0.2
total_telemetry_count = 0
total_reroute_event_count = 0

all_nodes = sorted(phase1.graph_adjacency.keys())
next_destination_by_origin = {}
for origin in all_nodes:
    choices = [node for node in all_nodes if node != origin]
    next_destination_by_origin[origin] = choices

def _roadwork_state_for_tick(tick: int) -> dict:
    state = {"active": False, "blocked_segment_ids": []}
    for event_tick in sorted(roadwork_events_by_tick.keys()):
        if event_tick <= tick:
            state = roadwork_events_by_tick[event_tick]
        else:
            break
    return state

def _effective_blocked_segments(tick: int) -> set[int]:
    effective = set(base_blocked_segments)
    roadwork_state = _roadwork_state_for_tick(tick)
    if bool(roadwork_state.get("active", False)):
        effective.update(int(segment_id) for segment_id in roadwork_state.get("blocked_segment_ids", []))
    effective.update(int(segment_id) for segment_id in live_congestion_state.get("congested_segment_ids", []))
    return effective

def _current_lnglat(car: dict) -> tuple[float, float]:
    route = car.get("route") or []
    index = int(car.get("route_step_index", 0))
    progress = float(car.get("segment_progress", 0.0))

    if route and index < len(route) - 1 and car.get("status") == "moving":
        from_node = route[index]
        to_node = route[index + 1]
        from_lng, from_lat = resolve_node_lnglat(from_node)
        to_lng, to_lat = resolve_node_lnglat(to_node)
        clamped_progress = min(max(progress, 0.0), 1.0)
        lng = from_lng + (to_lng - from_lng) * clamped_progress
        lat = from_lat + (to_lat - from_lat) * clamped_progress
        return lng, lat

    fallback_node = str(car.get("current_node", car["origin"]))
    return resolve_node_lnglat(fallback_node)

def _assign_next_trip(car: dict, effective_blocked: set[int], tick: int) -> None:
    origin = str(car.get("current_node", car["origin"]))
    choices = next_destination_by_origin.get(origin, [node for node in all_nodes if node != origin])
    car["trip_index"] = int(car.get("trip_index", 0)) + 1
    destination = choices[car["trip_index"] % len(choices)]

    new_route = compute_reroute(
        graph_adjacency=phase1.graph_adjacency,
        origin=origin,
        destination=destination,
        blocked_segments=effective_blocked,
        segment_node_pairs=phase1.segment_node_pairs,
    )

    car["origin"] = origin
    car["destination"] = destination
    car["planned_route"] = new_route
    car["route"] = new_route
    car["route_step_index"] = 0
    car["segment_progress"] = 0.1
    car["last_segment_id"] = None
    car["last_reroute_tick"] = tick

    if not new_route:
        car["status"] = "waiting"
    elif len(new_route) == 1:
        car["status"] = "arrived"
    else:
        car["status"] = "moving"
        car["current_node"] = new_route[0]

client = mqtt.connect_mqtt(config.mqtt, client_id_suffix="agent-cars-phase4")

def _subscribe_car_topics():
    client.subscribe(roadwork_topic)
    client.subscribe(congestion_topic)

def _on_message(client_obj, userdata, msg):
    payload = json.loads(msg.payload.decode("utf-8"))
    if msg.topic == roadwork_topic:
        event_tick = int(payload.get("tick", 0))
        roadwork_events_by_tick[event_tick] = {
            "active": bool(payload.get("active", False)),
            "blocked_segment_ids": [int(segment_id) for segment_id in payload.get("blocked_segment_ids", [])],
        }
    elif msg.topic == congestion_topic:
        live_congestion_state.update(payload)

_previous_on_connect = getattr(client, "on_connect", None)
def _on_connect(client_obj, userdata, flags, reason_code, properties=None):
    if callable(_previous_on_connect):
        _previous_on_connect(client_obj, userdata, flags, reason_code, properties)
    _subscribe_car_topics()

client.on_connect = _on_connect
client.on_message = _on_message
_subscribe_car_topics()

try:
    for tick in range(1, ticks_to_run + 1):
        tick_timestamp = datetime.now(timezone.utc).isoformat()
        telemetry_count = 0
        reroute_event_count = 0
        waiting = 0
        moving = 0
        arrived = 0

        effective_blocked = _effective_blocked_segments(tick)

        for car in cars:
            route = car.get("route") or []
            current_index = int(car.get("route_step_index", 0))

            if tick < int(car.get("start_tick", 0)):
                car["status"] = "waiting"
            else:
                if car["status"] == "arrived":
                    _assign_next_trip(car, effective_blocked, tick)
                    route = car.get("route") or []
                    current_index = int(car.get("route_step_index", 0))

                if car["status"] == "waiting" and route and current_index < len(route) - 1:
                    car["status"] = "moving"

                remaining_route = route[current_index:] if route else []
                intersects_blocked = bool(
                    remaining_route
                    and route_intersects_blocked_segments(
                        route_nodes=remaining_route,
                        blocked_segments=effective_blocked,
                        segment_node_pairs=phase1.segment_node_pairs,
                    )
                )
                cooldown_ready = (tick - int(car.get("last_reroute_tick", -999))) >= reroute_cooldown_ticks
                should_reroute = (not route) or intersects_blocked

                if should_reroute and cooldown_ready:
                    old_route = route
                    new_route = compute_reroute(
                        graph_adjacency=phase1.graph_adjacency,
                        origin=str(car.get("current_node", car["origin"])),
                        destination=car["destination"],
                        blocked_segments=effective_blocked,
                        segment_node_pairs=phase1.segment_node_pairs,
                    )

                    if new_route and new_route != old_route:
                        car["route"] = new_route
                        car["route_step_index"] = 0
                        car["segment_progress"] = 0.15
                        car["current_node"] = new_route[0]
                        car["last_reroute_tick"] = tick
                        car["status"] = "arrived" if len(new_route) == 1 else "moving"
                        car["was_rerouted"] = True

                        reroute_payload = {
                            "agent": "agent_cars",
                            "tick": tick,
                            "timestamp": tick_timestamp,
                            "car_id": car["car_id"],
                            "origin": car["origin"],
                            "destination": car["destination"],
                            "old_route": old_route,
                            "new_route": new_route,
                            "blocked_segment_ids": sorted(effective_blocked),
                        }
                        publish_json_checked(client, reroute_topic, reroute_payload)
                        reroute_event_count += 1
                        route = new_route
                        current_index = 0
                    elif not new_route:
                        car["status"] = "waiting"

                route = car.get("route") or []
                current_index = int(car.get("route_step_index", 0))

                if car["status"] == "moving" and route and current_index < len(route) - 1:
                    progress_step = max(0.12, float(car.get("speed_kmh", 24)) / 90.0)
                    segment_progress = float(car.get("segment_progress", 0.0)) + progress_step

                    while segment_progress >= 1.0 and current_index < len(route) - 1:
                        previous_node = route[current_index]
                        current_index += 1
                        reached_node = route[current_index]
                        segment_progress -= 1.0
                        car["current_node"] = reached_node
                        car["last_segment_id"] = segment_id_by_nodes.get((previous_node, reached_node))

                    car["route_step_index"] = current_index
                    car["segment_progress"] = segment_progress

                    if current_index >= len(route) - 1:
                        car["status"] = "arrived"
                        car["segment_progress"] = 0.0

            status = car["status"]
            current_node = str(car.get("current_node", car["origin"]))
            lng, lat = _current_lnglat(car)
            segment_id = car.get("last_segment_id")
            speed_kmh = int(car.get("speed_kmh", 0)) if status == "moving" else 0

            telemetry_payload = {
                "agent": "agent_cars",
                "tick": tick,
                "timestamp": tick_timestamp,
                "car_id": car["car_id"],
                "origin": car["origin"],
                "destination": car["destination"],
                "current_node": current_node,
                "current_lng": round(float(lng), 6),
                "current_lat": round(float(lat), 6),
                "segment_id": segment_id,
                "speed_kmh": speed_kmh,
                "status": status,
            }
            publish_json_checked(client, telemetry_topic, telemetry_payload)
            telemetry_count += 1

            if status == "waiting":
                waiting += 1
            elif status == "moving":
                moving += 1
            else:
                arrived += 1

        total_telemetry_count += telemetry_count
        total_reroute_event_count += reroute_event_count
        print(
            f"Tick {tick} publish complete: telemetry={telemetry_count}, reroute_events={reroute_event_count}, "
            f"waiting={waiting}, moving={moving}, arrived={arrived}"
        )
        time.sleep(tick_sleep_s)

    print(
        f"Published ticks={ticks_to_run}: telemetry={total_telemetry_count}, "
        f"reroute_events={total_reroute_event_count}"
    )
finally:
    client.loop_stop()
    client.disconnect()
    print("Disconnected MQTT client.")

Tick 1 publish complete: telemetry=10, reroute_events=0, waiting=6, moving=4, arrived=0
Tick 2 publish complete: telemetry=10, reroute_events=0, waiting=4, moving=6, arrived=0
Tick 3 publish complete: telemetry=10, reroute_events=0, waiting=2, moving=8, arrived=0
Tick 4 publish complete: telemetry=10, reroute_events=0, waiting=1, moving=8, arrived=1
Tick 5 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=10, arrived=0
Tick 6 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=9, arrived=1
Tick 7 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=10, arrived=0
Tick 8 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=8, arrived=2
Tick 9 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=7, arrived=3
Tick 10 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=7, arrived=3
Tick 11 publish complete: telemetry=10, reroute_events=0, waiting=0, moving=9, arrived=1
Tick 12 publish complete: te